In [1]:
!pip install groq pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00


In [15]:
from groq import Groq
import json
import yaml
from google.colab import userdata

# Add your Groq API key
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

MODEL = "llama-3.1-8b-instant"
TEMPERATURE = 0

In [3]:
def ask_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [4]:
json_prompt = """
Generate a JSON array containing information about three books.

Each object must contain only these keys:
"title", "author", "year"

Example format:
[
  {
    "title": "Book Name",
    "author": "Author Name",
    "year": 2000
  }
]

Rules:
- Output only valid JSON.
- Use double quotes.
- Do not use markdown.
- Do not add explanations.
- Do not include trailing commas.
"""

In [16]:
response = ask_llm(json_prompt)

print("=== INITIAL RESPONSE ===")
print(response)


def validate_json(output):
    try:
        data = json.loads(output)

        # Check array
        if not isinstance(data, list):
            return False, "Output is not a JSON array"

        # Check exactly 3 books
        if len(data) != 3:
            return False, "JSON does not contain exactly 3 books"

        required_keys = {"title", "author", "year"}

        for book in data:
            if set(book.keys()) != required_keys:
                return False, "Missing or extra keys found"

        return True, "Valid JSON"

    except json.JSONDecodeError as e:
        return False, str(e)

=== INITIAL RESPONSE ===
[
  {
    "title": "Book 1",
    "author": "Author 1",
    "year": 1999
  },
  {
    "title": "Book 2",
    "author": "Author 2",
    "year": 2001
  },
  {
    "title": "Book 3",
    "author": "Author 3",
    "year": 2002
  }
]


In [17]:
valid, message = validate_json(response)

print("\nValidation Result:")
print(message)


if not valid:

    print("\nRetrying with refined prompt...")

    refined_prompt = """
Output ONLY valid JSON.

Create an array of exactly 3 book objects.

Required keys:
"title",
"author",
"year"

Rules:
- No markdown.
- No explanation.
- No comments.
- Use double quotes.
- No trailing commas.

Return only JSON.
"""

    response = ask_llm(refined_prompt)

    print("\n=== REFINED RESPONSE ===")
    print(response)

    valid, message = validate_json(response)

    print("\nFinal Validation:")
    print(message)


Validation Result:
Valid JSON


In [18]:
try:
    data = json.loads(response)

    yaml_output = yaml.dump(
        data,
        sort_keys=False
    )

    print("\n=== YAML OUTPUT ===")
    print(yaml_output)

    yaml_data = yaml.safe_load(yaml_output)

    print("YAML Validation: Successful")

except Exception as e:
    print("YAML Error:", e)


=== YAML OUTPUT ===
- title: Book 1
  author: Author 1
  year: 1999
- title: Book 2
  author: Author 2
  year: 2001
- title: Book 3
  author: Author 3
  year: 2002

YAML Validation: Successful
